Chunking of Documents

In [3]:
#imports
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os

g:\Python Projects\2 AI concepts and patterns\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#preparing a text splitters

char_splitter = RecursiveCharacterTextSplitter(chunk_size=128,chunk_overlap=0)

In [5]:
directory = "./Documents"
fileNames = os.listdir(directory)
chunks = []
for file in fileNames:
    with open(f"{directory}/{file}") as f:
        filetext = f.read()
        texts = char_splitter.split_text(filetext)
        for chunkindex,text in enumerate(texts):
            chunks.append(Document(page_content=text,metadata={
                "chunkindex":chunkindex,
                "sourcefile":file
            }))

In [6]:
for chunk in chunks[:2]:
    print("------------")
    print("Page Content")
    print(chunk.page_content)
    print("Metadata")
    print(chunk.metadata)

------------
Page Content
# Development Team Internal Directory

## Team Scope

Handles:
Metadata
{'chunkindex': 0, 'sourcefile': 'Development Team.txt'}
------------
Page Content
* Feature development (frontend & backend)
* Bug fixing and issue resolution
* API development and integration
Metadata
{'chunkindex': 1, 'sourcefile': 'Development Team.txt'}


In [7]:
# if embeddings:
#     del embeddings
embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3457.05it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vector_store = FAISS.from_documents(documents=chunks,embedding=embeddings)

In [9]:
# retriever = vector_store.as_retriever(top=10)

In [11]:
#ReRanker
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Your base retriever (e.g., FAISS)
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Initialize BGE reranker
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=3)

# Wrap retriever
reranker = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Retrieve
docs = reranker.invoke("How much years of experience Ankit Sharma have?")   

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3515.65it/s]


In [12]:
docs

[Document(id='a10222ac-58ef-4134-9800-e99b95d017e9', metadata={'chunkindex': 9, 'sourcefile': 'Development Team.txt'}, page_content='---\n\n### 2. Ankit Sharma'),
 Document(id='6714f371-33c2-4c95-bd47-cce707d7864b', metadata={'chunkindex': 35, 'sourcefile': 'IT Support Team.txt'}, page_content='Contact: ankit.mehta@company.com\nPriority Level: High\n10. Pooja Deshmukh\nRole: Junior IT Support\nExperience: 2 years'),
 Document(id='865a4ec1-060a-4bdb-8452-e617ee9279a0', metadata={'chunkindex': 1, 'sourcefile': 'IT Support Team.txt'}, page_content='Team Members (Rich Data for RAG)\n1. Rahul Sharma\nRole: Senior IT Support Engineer\nExperience: 8 years')]

In [13]:
def get_surrounding_chunks(retrieved_chunk, all_chunks, window=5):
    idx = retrieved_chunk.metadata["chunkindex"]
    start = max(0, idx - window)
    end = min(len(all_chunks), idx + window + 1)
    return all_chunks[start:end]

In [ ]:
# retrieved_chunks=retriever.invoke("How much years of experience Ankit Sharma have?")

In [14]:
context_chunks = []
for retrievedchunk in docs:
    context_chunks = context_chunks + get_surrounding_chunks(retrievedchunk,chunks)

In [18]:
print(len(context_chunks))
contextText = ""
contextText.join([f"Source:{content.metadata["sourcefile"]}, content : {content.page_content}" for content in context_chunks])

29


'Source:Development Team.txt, content : * Role: Senior Frontend Developer\n* Experience: 7 years\n* Expertise: React, UI performance, state managementSource:Development Team.txt, content : * Skill Level: Expert\n* Primary Keywords: UI bug, frontend issue, component not rendering, state issueSource:Development Team.txt, content : * Secondary Skills: accessibility, responsive design\n* Tools Used: React, Redux, Chrome DevToolsSource:Development Team.txt, content : * Desk Location: Floor 4, Bay F12\n* Availability: 10 AM â€“ 6 PMSource:Development Team.txt, content : * Contact: [rohan.mehta@company.com](mailto:rohan.mehta@company.com)\n* Priority Level: HighSource:Development Team.txt, content : ---\n\n### 2. Ankit SharmaSource:Development Team.txt, content : * Role: Backend Developer\n* Experience: 5 years\n* Expertise: Node.js, APIs, database handling\n* Skill Level: ExpertSource:Development Team.txt, content : * Primary Keywords: API error, backend issue, server error, database issue\n

In [ ]:
del embeddings
del model
# import torch
# print(torch.cuda.is_available()) 

False
